# Módulo 05: Data Wrangling (Junção, Combinação e Remodelação)
>
> [Universidade Federal do Ceará (UFC)](https://www.ufc.br/)\
> [Departamento de Computação (DC)](https://dc.ufc.br/pt/)\
> [Capacitação Técnica e Empreendedora em IA (CTE-IA)](https://www.cteia.dc.ufc.br/)\
> Fase I: Capacitação Teórica de IA\
> Disciplina: Programação para Ciência de Dados (60h)\
> Professor: [Lincoln S. Rocha](http://lattes.cnpq.br/0656977742590515)\
> E-mail: <lincoln@dc.ufc.br>
>

Em muitas aplicações, os dados podem estar dispersos por vários arquivos ou bancos de dados, ou organizados de forma que não seja conveniente para análise. Este módulo se concentra em ferramentas que auxiliam na combinação, junção e reorganização de dados Inicialmente, é apresentado o conceito de indexação hierárquica no Pandas. Em seguida, exploraremos as manipulações de dados específicas. Este módulo cobrirá o seguinte conteúdo:

1. Indexação Hierárquica
2. Combinação e Junção de Datasets
3. Remodelagem e Rotação

## 1. Indexação Hierárquica

### 1.1. Visão Geral

A indexação hierárquica é um recurso importante do Pandas que permite ter vários (dois ou mais) níveis de índice em um eixo. Outra forma de pensar nisso é que ela oferece uma maneira de trabalhar com dados de alta dimensionalidade em um formato de baixa dimensionalidade. Vamos começar com um exemplo simples: crie uma `Series` com uma lista de listas (ou arrays) como índice:

In [ ]:
import numpy as np
import pandas as pd

data = pd.Series(np.random.uniform(size=9),
                 index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])

data

a  1    0.022388
   2    0.829412
   3    0.580918
b  1    0.908917
   3    0.033992
c  1    0.435822
   2    0.492348
d  2    0.829506
   3    0.825101
dtype: float64

O que você está vendo é uma visualização simplificada de uma `Series` com um `MultiIndex` como índice. Os "espaços" na exibição do índice significam "use o rótulo diretamente acima":

In [2]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

Com um objeto indexado hierarquicamente, é possível a chamada indexação parcial, que permite selecionar subconjuntos dos dados de forma concisa:

In [3]:
data['b']

1    0.908917
3    0.033992
dtype: float64

In [4]:
data['b': 'c']

b  1    0.908917
   3    0.033992
c  1    0.435822
   2    0.492348
dtype: float64

In [6]:
data.loc[['b', 'd']]

b  1    0.908917
   3    0.033992
d  2    0.829506
   3    0.825101
dtype: float64

A seleção é possível até mesmo a partir de um nível `"interno"`. Aqui, selecionamos todos os valores que possuem o valor `2` a partir do segundo nível do índice:

In [7]:
data.loc[:, 2]

a    0.829412
c    0.492348
d    0.829506
dtype: float64

A indexação hierárquica desempenha um papel importante na remodelagem de dados e em operações baseadas em grupos, como a criação de uma tabela dinâmica. Por exemplo, você pode reorganizar esses dados em um `DataFrame` usando o método `unstack`:

In [8]:
data.unstack()

,1,2,3
a,0.022388,0.829412,0.580918
b,0.908917,NaN,0.033992
c,0.435822,0.492348,NaN
d,NaN,0.829506,0.825101


A operação inversa de `unstack` é `stack`:

In [9]:
data.unstack().stack()

a  1    0.022388
   2    0.829412
   3    0.580918
b  1    0.908917
   2         NaN
   3    0.033992
c  1    0.435822
   2    0.492348
   3         NaN
d  1         NaN
   2    0.829506
   3    0.825101
dtype: float64

Os métodos `stack` e `unstack` serão explorados em mais detalhes na seção `Remodelagem e Rotação`.

Em um DataFrame, qualquer um dos eixos pode ter um índice hierárquico:

In [10]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                     columns=[['Ohio', 'Ohio', 'Colorado'],
                              ['Green', 'Red', 'Green']])

frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

Os níveis hierárquicos podem ter nomes (como strings ou quaisquer objetos Python). Nesse caso, eles aparecerão na saída do console:

In [11]:
frame.index.names = ['key1', 'key2']
frame.columns.names = ['state', 'color']

frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

Esses nomes substituem o atributo `name`, que é usado apenas com índices de nível único.

Você pode ver quantos níveis um índice possui acessando seu atributo `nlevels`:

In [12]:
frame.index.nlevels

2

Com a indexação parcial de colunas, você pode selecionar grupos de colunas da mesma forma:

In [13]:
frame['Ohio']

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

Um `MultiIndex` pode ser criado individualmente e reutilizado; as colunas no `DataFrame` anterior com nomes de nível também poderiam ser criadas desta forma:

In [14]:
pd.MultiIndex.from_arrays([['Ohio', 'Ohio', 'Colorado'],
                           ['Green', 'Red', 'Green']],
                           names=['state', 'color'])

MultiIndex([(    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

### 1.2. Reordenando e Classificando os Níveis

Em alguns casos, pode ser necessário reorganizar a ordem dos níveis em um eixo ou classificar os dados pelos valores em um nível específico. O método `swaplevel` recebe dois números ou nomes de nível e retorna um novo objeto com os níveis trocados (mas os dados permanecem inalterados):

In [15]:
frame.swaplevel('key1', 'key2')

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

Por padrão, a função `sort_index` ordena os dados lexicograficamente usando todos os níveis de índice, mas você pode optar por usar apenas um único nível ou um subconjunto de níveis para ordenar, passando o argumento `level`. Por exemplo:

In [16]:
frame.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [17]:
frame.swaplevel(0, 1).sort_index(level=0)

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

>
> O desempenho da seleção de dados é muito melhor em objetos indexados hierarquicamente se o índice for classificado lexicograficamente a partir do nível mais externo — isto é, o resultado de chamar `sort_index(level=0)` ou `sort_index()`.
>

### 1.3. Estatísticas Resumidas por Nível

Muitas estatísticas descritivas e de resumo em `DataFrames` e `Series` possuem uma opção de `level` na qual você pode especificar o nível pelo qual deseja agregar em um eixo específico. Considere o `DataFrame` anterior; podemos agregar por nível nas linhas ou nas colunas, da seguinte forma:

In [18]:
frame.groupby(level='key2').sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

In [19]:
# frame.groupby(level="color", axis="columns").sum() # deprecated
frame.T.groupby(level='color').sum().T

color      Green  Red
key1 key2            
a    1         2    1
     2         8    4
b    1        14    7
     2        20   10

Discutiremos `groupby` com mais detalhes posteriormente.

### 1.4. Indexação com as Colunas de um DataFrame

Não é incomum querer usar uma ou mais colunas de um `DataFrame` como índice de linha; alternativamente, você pode querer mover o índice de linha para as colunas do `DataFrame`. Aqui está um exemplo de `DataFrame`:

In [20]:
frame = pd.DataFrame({'a': range(7), 'b': range(7, 0, -1),
                      'c': ['one', 'one', 'one', 'two', 'two', 'two', 'two'],
                      'd': [0, 1, 2, 0, 1, 2, 3]})

frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


A função `set_index` do `DataFrame` criará um novo `DataFrame` usando uma ou mais de suas colunas como índice:

In [21]:
frame2 = frame.set_index(['c', 'd'])

frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

Por padrão, as colunas são removidas do `DataFrame`, mas você pode mantê-las passando `drop=False` para `set_index`:

In [22]:
frame.set_index(['c', 'd'], drop=False)

a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

O comando `reset_index`, por outro lado, faz o oposto do `set_index`; os níveis do índice hierárquico são movidos para as colunas:

In [23]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


## 2. Combinação e Junção de Datasets

### 2.1. Visão Geral

Os dados contidos em objetos pandas podem ser combinados de diversas maneiras:

- `pandas.merge`: Conecta linhas em `DataFrames` com base em uma ou mais chaves. Isso será familiar para usuários de `SQL` ou outros bancos de dados relacionais, pois implementa operações de *join* de banco de dados.

- `pandas.concat`: Concatena ou `"empilha"` objetos ao longo de um eixo.

- `combine_first`: Combina dados sobrepostos para preencher valores ausentes em um objeto com valores de outro.

Abordaremos cada um desses pontos e daremos vários exemplos.

### 2.2. Junções de DataFrames no Estilo de Banco de Dados

As operações de mesclagem (*merge*) ou junção (*join*) combinam conjuntos de dados vinculando linhas por meio de uma ou mais chaves. Essas operações são particularmente importantes em bancos de dados relacionais (por exemplo, baseados em SQL). A função `pandas.merge` do Pandas é o principal ponto de entrada para usar esses algoritmos em seus dados.

In [ ]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': pd.Series(range(7), dtype='Int64')})

df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [3]:
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],
                    'data2': pd.Series(range(3), dtype='Int64')})

df2

,key,data2
0,a,0
1,b,1
2,d,2


Aqui estamos usando o tipo de extensão `Int64` do Pandas para inteiros `"anuláveis"`, discutido no `Módulo 4`.

Este é um exemplo de uma junção `muitos-para-um`(*many-to-one*); os dados em `df1` têm várias linhas rotuladas como `a` e `b`, enquanto `df2` tem apenas uma linha para cada valor na coluna `key`. Chamando `pandas.merge` com esses objetos, obtemos:

In [4]:
pd.merge(df1, df2)

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


Observe que não especificamos qual coluna usar para a junção (*join*). Se essa informação não for especificada, o `pandas.merge` usa os nomes das colunas sobrepostas como chaves. No entanto, é uma boa prática especificar explicitamente:

In [ ]:
pd.merge(df1, df2, on='key')

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


Em geral, a ordem de saída das colunas nas operações de `pandas.merge` não é especificada.

Se os nomes das colunas forem diferentes em cada objeto, você pode especificá-los separadamente:

In [6]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': pd.Series(range(7), dtype='Int64')})

df3

,lkey,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [7]:
df4 = pd.DataFrame({'rkey': ['a', 'b', 'd'],
                    'data2': pd.Series(range(3), dtype='Int64')})

df4

,rkey,data2
0,a,0
1,b,1
2,d,2


In [9]:
pd.merge(df3, df4, left_on='lkey', right_on='rkey')

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,a,2,a,0
3,a,4,a,0
4,a,5,a,0
5,b,6,b,1


Você pode notar que os valores `"c"` e `"d"` e os dados associados estão ausentes do resultado. Por padrão, o `pandas.merge` realiza uma junção `"interna"`(`inner` *join*); as chaves no resultado são a interseção, ou seja, o conjunto comum encontrado em ambas as tabelas. Outras opções possíveis são `"left"`, `"rigth"` e `"outer"`. A junção externa (`outer` *join*) realiza a união das chaves, combinando o efeito da aplicação das junções esquerda (`left` *join*) e direita (`rigth` *join*):

In [10]:
pd.merge(df1, df2, how='outer')

,key,data1,data2
0,a,2,0
1,a,4,0
2,a,5,0
3,b,0,1
4,b,1,1
5,b,6,1
6,c,3,<NA>
7,d,<NA>,2


In [11]:
pd.merge(df3, df4, left_on='lkey', right_on='rkey', how='outer')

,lkey,data1,rkey,data2
0,a,2,a,0
1,a,4,a,0
2,a,5,a,0
3,b,0,b,1
4,b,1,b,1
5,b,6,b,1
6,c,3,NaN,<NA>
7,NaN,<NA>,d,2


Em uma junção externa (`outer` *join*), as linhas dos objetos `DataFrame` da esquerda ou da direita que não correspondem às chaves do outro `DataFrame` aparecerão com valores `NA` nas colunas do outro `DataFrame` para as linhas não correspondentes.

Consulte a tabela abaixo para obter um resumo das opções de `how`.

| Opção | Comportamento |
|--------|----------------|
| `how="inner"` | Usa apenas as combinações de chaves que aparecem em **ambas** as tabelas. |
| `how="left"` | Usa todas as combinações de chaves encontradas na tabela **da esquerda**. |
| `how="right"` | Usa todas as combinações de chaves encontradas na tabela **da direita**. |
| `how="outer"` | Usa todas as combinações de chaves observadas em **ambas** as tabelas, unindo os resultados. |

As combinações muitos-para-muitos (*many-to-many*) formam o produto cartesiano das chaves correspondentes. Aqui está um exemplo.

In [12]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],
                    'data1': pd.Series(range(6), dtype='Int64')})

df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [13]:
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'],
                    'data2': pd.Series(range(5), dtype='Int64')})

df2

,key,data2
0,a,0
1,b,1
2,a,2
3,b,3
4,d,4


In [14]:
pd.merge(df1, df2, on='key', how='left')

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,c,3,<NA>
7,a,4,0
8,a,4,2
9,b,5,1


Como havia três linhas `"b"` no `DataFrame` da esquerda e duas no da direita, o resultado terá seis linhas `"b"`. O método de junção passado para o argumento de palavra-chave `how` afeta apenas os valores de chave distintos que aparecem no resultado:

In [15]:
pd.merge(df1, df2, how='inner')

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,a,4,0
7,a,4,2
8,b,5,1
9,b,5,3


Para mesclar com várias chaves, passe uma lista de nomes de colunas:

In [16]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],
                     'key2': ['one', 'two', 'one'],
                     'lval': pd.Series([1, 2, 3], dtype='Int64')})
left

,key1,key2,lval
0,foo,one,1
1,foo,two,2
2,bar,one,3


In [17]:
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],
                      'key2': ['one', 'one', 'one', 'two'],
                      'rval': pd.Series([4, 5, 6, 7], dtype='Int64')})
right

,key1,key2,rval
0,foo,one,4
1,foo,one,5
2,bar,one,6
3,bar,two,7


In [19]:
pd.merge(left, right, on=['key1', 'key2'], how='outer')

,key1,key2,lval,rval
0,bar,one,3,6
1,bar,two,<NA>,7
2,foo,one,1,4
3,foo,one,1,5
4,foo,two,2,<NA>


Para determinar quais combinações de chaves aparecerão no resultado, dependendo do método de mesclagem escolhido, pense nas múltiplas chaves como formando uma matriz de tuplas a serem usadas como uma única chave de junção.

>
> Nota! Ao unir colunas em colunas, os índices dos objetos DataFrame passados ​​são descartados. Se precisar preservar os valores dos índices, você pode usar `reset_index` para adicionar o índice às colunas.
>

Uma última questão a ser considerada nas operações de mesclagem é o tratamento de nomes de colunas sobrepostos. Por exemplo:

In [20]:
pd.merge(left, right, on='key1')

,key1,key2_x,lval,key2_y,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


Embora seja possível resolver a sobreposição manualmente, o `pandas.merge` possui uma opção `suffixes` para especificar strings a serem anexadas aos nomes sobrepostos nos objetos `DataFrame` da esquerda e da direita:

In [21]:
pd.merge(left, right, on='key1', suffixes=('_left', '_right'))

,key1,key2_left,lval,key2_right,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


Consulte a tabela abaixo para obter uma referência dos argumentos do `pandas.merge`.

| Argumento | Descrição |
|------------|------------|
| `left` | DataFrame a ser mesclado (merge) no lado esquerdo. |
| `right` | DataFrame a ser mesclado (merge) no lado direito. |
| `how` | Tipo de junção (join) a ser aplicada: pode ser `"inner"`, `"outer"`, `"left"` ou `"right"`; o padrão é `"inner"`. |
| `on` | Nomes das colunas usadas para realizar o join. Devem estar presentes em ambos os objetos `DataFrame`. Se não for especificado e nenhuma outra chave de junção for informada, será usada a interseção dos nomes de colunas em `left` e `right`. |
| `left_on` | Colunas do `DataFrame` da esquerda (`left`) usadas como chaves de junção. Pode ser um único nome de coluna ou uma lista de nomes. |
| `right_on` | Análogo a `left_on`, mas para o `DataFrame` da direita (`right`). |
| `left_index` | Usa o índice de linhas do `DataFrame` da esquerda (`left`) como chave de junção (ou chaves, no caso de `MultiIndex`). |
| `right_index` | Análogo a `left_index`, mas para o `DataFrame` da direita (`right`). |
| `sort` | Ordena os dados mesclados de forma lexicográfica pelas chaves de junção; o padrão é `False`. |
| `suffixes` | Tupla de valores de string adicionados aos nomes de colunas em caso de sobreposição; o padrão é `("_x", "_y")`. Exemplo: se ambas as tabelas tiverem uma coluna `"data"`, o resultado terá `"data_x"` e `"data_y"`. |
| `copy` | Se `False`, evita copiar os dados para a estrutura resultante em alguns casos excepcionais; por padrão, os dados são sempre copiados. |
| `validate` | Verifica se o tipo de junção é o especificado (por exemplo, um-para-um, um-para-muitos ou muitos-para-muitos). Consulte a documentação para mais detalhes sobre as opções. |
| `indicator` | Adiciona uma coluna especial `_merge` que indica a origem de cada linha: `"left_only"`, `"right_only"` ou `"both"`, conforme a origem dos dados unidos em cada linha. |

### 2.3. Fusão no Índice

Em alguns casos, a(s) chave(s) de mesclagem em um `DataFrame` serão encontradas em seu índice (rótulos das linhas). Nesse caso, você pode passar `left_index=True` ou `right_index=True` (ou ambos) para indicar que o índice deve ser usado como chave de mesclagem:

In [22]:
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'],
                      'value': pd.Series(range(6), dtype='Int64')})

left1

,key,value
0,a,0
1,b,1
2,a,2
3,a,3
4,b,4
5,c,5


In [24]:
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])

right1

,group_val
a,3.5
b,7.0


In [25]:
pd.merge(left1, right1, left_on='key', right_index=True)

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0


>
> Nota! Se você observar atentamente, verá que os valores de índice para `left1` foram preservados, enquanto em outros exemplos acima, os índices dos objetos `DataFrame` de entrada são descartados. Como o índice de `right1` é único, essa mesclagem `"many-to-one"` (com o método padrão `how="inner"`) pode preservar os valores de índice de `left1` que correspondem às linhas na saída.
>

Como o método de mesclagem padrão é a interseção das chaves de junção, você pode, em vez disso, formar a união delas com uma junção externa (*outer join*):

In [26]:
pd.merge(left1, right1, left_on='key', right_index=True, how='outer')

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0
5,c,5,NaN


Com dados indexados hierarquicamente, as coisas são mais complicadas, pois a junção por índice é equivalente a uma mesclagem de múltiplas chaves:

In [ ]:
lefth = pd.DataFrame({'key1': ['Ohio', 'Ohio', 'Ohio', 'Nevada', 'Nevada'],
                      'key2': [2000, 2001, 2002, 2001, 2002],
                      'data': pd.Series(range(5), dtype='Int64')})

lefth

,key1,key2,data
0,Ohio,2000,0
1,Ohio,2001,1
2,Ohio,2002,2
3,Nevada,2001,3
4,Nevada,2002,4


In [30]:
righth_index = pd.MultiIndex.from_arrays([['Nevada', 'Nevada', 'Ohio', 'Ohio', 'Ohio', 'Ohio'],
                                          [2000, 2001, 2000, 2000, 2001, 2002]])

In [31]:
righth = pd.DataFrame({'event1': pd.Series([0, 2, 4, 6, 8, 10], dtype='Int64', index=righth_index),
                       'event2': pd.Series([1, 3, 5, 7, 9, 11], dtype='Int64', index=righth_index)})

righth

event1  event2
Nevada 2000       0       1
       2001       2       3
Ohio   2000       4       5
       2000       6       7
       2001       8       9
       2002      10      11

Neste caso, você precisa indicar várias colunas para mesclar como uma lista (observe o tratamento de valores de índice duplicados com `how="outer"`):

In [32]:
pd.merge(lefth, righth, left_on=['key1', 'key2'], right_index=True)

,key1,key2,data,event1,event2
0,Ohio,2000,0,4,5
0,Ohio,2000,0,6,7
1,Ohio,2001,1,8,9
3,Nevada,2001,3,2,3
2,Ohio,2002,2,10,11


In [33]:
pd.merge(lefth, righth, left_on=['key1', 'key2'], right_index=True, how='outer')

,key1,key2,data,event1,event2
4,Nevada,2000,<NA>,0,1
3,Nevada,2001,3,2,3
4,Nevada,2002,4,<NA>,<NA>
0,Ohio,2000,0,4,5
0,Ohio,2000,0,6,7
1,Ohio,2001,1,8,9
2,Ohio,2002,2,10,11


Também é possível usar os índices de ambos os lados da mesclagem:

In [34]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]], 
                     index=['a', 'c', 'e'], 
                     columns=['Ohio', 'Nevada']).astype('Int64')

left2

,Ohio,Nevada
a,1,2
c,3,4
e,5,6


In [35]:
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                       index=['b', 'c', 'd', 'e'],
                       columns=['Missouri', 'Alabama']).astype('Int64')

right2

,Missouri,Alabama
b,7,8
c,9,10
d,11,12
e,13,14


In [36]:
pd.merge(left2, right2, how='outer', left_index=True, right_index=True)

,Ohio,Nevada,Missouri,Alabama
a,1,2,<NA>,<NA>
b,<NA>,<NA>,7,8
c,3,4,9,10
d,<NA>,<NA>,11,12
e,5,6,13,14


O `DataFrame` possui um método de instância `join` para simplificar a mesclagem por índice. Ele também pode ser usado para combinar vários objetos `DataFrame` que possuem índices iguais ou semelhantes, mas colunas não sobrepostas. No exemplo anterior, poderíamos ter escrito:

In [37]:
left2.join(right2, how='outer')

,Ohio,Nevada,Missouri,Alabama
a,1,2,<NA>,<NA>
b,<NA>,<NA>,7,8
c,3,4,9,10
d,<NA>,<NA>,11,12
e,5,6,13,14


Em comparação com o `pandas.merge`, o método `join` do `DataFrame` realiza uma junção à esquerda (`left` *join*) nas chaves de junção por padrão. Ele também suporta a junção do índice do `DataFrame` passado em uma das colunas do `DataFrame` que fez a chamada:

In [38]:
left1.join(right1, on='key')

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0
5,c,5,NaN


Você pode pensar nesse método como a junção de dados `"into"` objeto cujo método `join` foi chamado.

Por fim, para mesclagens simples de índices, você pode passar uma lista de `DataFrames` para unir (`join`) como alternativa ao uso da função `pandas.concat` mais geral descrita na próxima seção:

In [39]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=['a', 'c', 'e', 'f'],
                       columns=['New York', 'Oregon'])

another

,New York,Oregon
a,7.0,8.0
c,9.0,10.0
e,11.0,12.0
f,16.0,17.0


In [40]:
left2.join([right2, another])

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1,2,<NA>,<NA>,7.0,8.0
c,3,4,9,10,9.0,10.0
e,5,6,13,14,11.0,12.0


In [41]:
left2.join([right2, another], how='outer')

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1,2,<NA>,<NA>,7.0,8.0
b,<NA>,<NA>,7,8,NaN,NaN
c,3,4,9,10,9.0,10.0
d,<NA>,<NA>,11,12,NaN,NaN
e,5,6,13,14,11.0,12.0
f,<NA>,<NA>,<NA>,<NA>,16.0,17.0


### 2.4. Concatenando ao Longo de um Eixo

Outro tipo de operação de combinação de dados é referido de forma intercambiável como concatenação ou empilhamento. A função `concatenate` do NumPy pode fazer isso com arrays NumPy:

In [49]:
arr = np.arange(12).reshape((3, 4))

arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [50]:
np.concatenate([arr, arr], axis=1)

array([[ 0,  1,  2,  3,  0,  1,  2,  3],
       [ 4,  5,  6,  7,  4,  5,  6,  7],
       [ 8,  9, 10, 11,  8,  9, 10, 11]])

No contexto de objetos do Pandas, como `Series` e `DataFrame`, ter eixos rotulados permite generalizar ainda mais a concatenação de arrays. Em particular, surgem algumas preocupações adicionais:

- Se os objetos forem indexados de forma diferente nos outros eixos, devemos combinar os elementos distintos nesses eixos ou usar apenas os valores em comum?

- Os blocos de dados concatenados precisam ser identificáveis ​​como tal no objeto resultante?

- O `"eixo de concatenação"` contém dados que precisam ser preservados? Em muitos casos, os rótulos inteiros padrão em um `DataFrame` são melhor descartados durante a concatenação.

A função `concat` do Pandas oferece uma maneira consistente de abordar cada uma dessas questões. Veremos alguns exemplos para ilustrar como ela funciona. Suponha que temos três `Series` sem sobreposição de índices:

In [51]:
s1 = pd.Series([0, 1], index=['a', 'b'], dtype='Int64')

s1

a    0
b    1
dtype: Int64

In [52]:
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'], dtype='Int64')

s2

c    2
d    3
e    4
dtype: Int64

In [53]:
s3 = pd.Series([5, 6], index=['f', 'g'], dtype='Int64')

s3

f    5
g    6
dtype: Int64

Chamar `pandas.concat` com esses objetos em uma lista une os valores e índices:

In [54]:
pd.concat([s1, s2, s3])

a    0
b    1
c    2
d    3
e    4
f    5
g    6
dtype: Int64

Por padrão, o `pandas.concat` funciona ao longo do eixo `"index"`, produzindo outra `Series`. Se você passar o eixo `"columns"`, o resultado será um `DataFrame`:

In [55]:
pd.concat([s1, s2, s3], axis='columns')

,0,1,2
a,0,<NA>,<NA>
b,1,<NA>,<NA>
c,<NA>,2,<NA>
d,<NA>,3,<NA>
e,<NA>,4,<NA>
f,<NA>,<NA>,5
g,<NA>,<NA>,6


Neste caso, não há sobreposição no outro eixo, que, como você pode ver, é a união (a `outer` *join*) dos índices. Em vez disso, você pode intersectá-los passando `join="inner"`:

In [58]:
s4 = pd.concat([s1, s3])

s4

a    0
b    1
f    5
g    6
dtype: Int64

In [57]:
pd.concat([s1, s4], axis='columns')

,0,1
a,0,0
b,1,1
f,<NA>,5
g,<NA>,6


In [59]:
pd.concat([s1, s4], axis='columns', join='inner')

,0,1
a,0,0
b,1,1


Neste último exemplo, os rótulos `"f"` e `"g"` desapareceram devido à opção `join="inner"`.

Um possível problema é que as partes concatenadas não sejam identificáveis ​​no resultado. Suponha que, em vez disso, você queira criar um índice hierárquico no eixo de concatenação. Para fazer isso, use o argumento `keys`:

In [60]:
result = pd.concat([s1, s1, s3], keys=['one', 'two', 'three'])

result

one    a    0
       b    1
two    a    0
       b    1
three  f    5
       g    6
dtype: Int64

In [61]:
result.unstack()

,a,b,f,g
one,0,1,<NA>,<NA>
two,0,1,<NA>,<NA>
three,<NA>,<NA>,5,6


No caso de combinar `Series` ao longo de `axis="columns"`, as chaves se tornam os cabeçalhos das colunas do `DataFrame`:

In [62]:
pd.concat([s1, s2, s3], axis='columns', keys=['one', 'two', 'three'])

,one,two,three
a,0,<NA>,<NA>
b,1,<NA>,<NA>
c,<NA>,2,<NA>
d,<NA>,3,<NA>
e,<NA>,4,<NA>
f,<NA>,<NA>,5
g,<NA>,<NA>,6


A mesma lógica se aplica a objetos `DataFrame`:

In [63]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), 
                   index=['a', 'b', 'c'],
                   columns=['one', 'two'])

df1

,one,two
a,0,1
b,2,3
c,4,5


In [64]:
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), 
                   index=['a', 'c'],
                   columns=['three', 'four'])

df2

,three,four
a,5,6
c,7,8


In [65]:
pd.concat([df1, df2], axis='columns', keys=['level1', 'level2'])

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

Aqui, o argumento `keys` é usado para criar um índice hierárquico onde o primeiro nível pode ser usado para identificar cada um dos objetos `DataFrame` concatenados.

Se você passar um dicionário de objetos em vez de uma lista, as chaves do dicionário serão usadas para a opção `keys`.

In [66]:
pd.concat({'level1': df1, 'level2': df2}, axis='columns')

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

Existem argumentos adicionais que regem a forma como o índice hierárquico é criado (veja tabela a seguir). Por exemplo, podemos nomear os níveis do eixo criado com o argumento `names`:

In [67]:
pd.concat([df1, df2], axis='columns', 
          keys=['level1', 'level2'],
          names=['upper', 'lower'])

upper level1     level2     
lower    one two  three four
a          0   1    5.0  6.0
b          2   3    NaN  NaN
c          4   5    7.0  8.0

Uma última consideração diz respeito aos `DataFrames` em que o índice da linha não contém quaisquer dados relevantes:

In [68]:
df1 = pd.DataFrame(np.random.standard_normal((3, 4)),
                   columns=['a', 'b', 'c', 'd'])

df1

,a,b,c,d
0,-0.729818,-0.361646,0.448201,-0.566798
1,-0.505771,0.816148,-0.332647,0.654787
2,0.205649,0.924114,0.794957,0.223813


In [69]:
df2 = pd.DataFrame(np.random.standard_normal((2, 3)),
                   columns=['b', 'd', 'a'])

df2

,b,d,a
0,0.813568,0.061397,1.177531
1,0.520181,-0.546758,-0.090410


Nesse caso, você pode passar `ignore_index=True`, que descarta os índices de cada `DataFrame` e concatena os dados apenas nas colunas, atribuindo um novo índice padrão:

In [70]:
pd.concat([df1, df2], ignore_index=True)

,a,b,c,d
0,-0.729818,-0.361646,0.448201,-0.566798
1,-0.505771,0.816148,-0.332647,0.654787
2,0.205649,0.924114,0.794957,0.223813
3,1.177531,0.813568,NaN,0.061397
4,-0.090410,0.520181,NaN,-0.546758


A tabela abaixo descreve os argumentos da função `pandas.concat`.

| Argumento | Descrição |
|------------|------------|
| `objs` | Lista ou dicionário de objetos do pandas a serem concatenados; este é o único argumento obrigatório. |
| `axis` | Eixo ao longo do qual a concatenação será feita; por padrão, concatena ao longo das linhas (`axis="index"`). |
| `join` | Pode ser `"inner"` ou `"outer"` (padrão é `"outer"`); define se os índices devem ser intersectados (`inner`) ou unidos (`outer`) nos outros eixos. |
| `keys` | Valores associados aos objetos concatenados, formando um índice hierárquico ao longo do eixo de concatenação; pode ser uma lista ou array de valores arbitrários, tuplas ou lista de arrays (no caso de múltiplos níveis definidos em `levels`). |
| `levels` | Índices específicos a serem usados como níveis hierárquicos, caso `keys` seja informado. |
| `names` | Nomes atribuídos aos níveis hierárquicos criados, caso `keys` e/ou `levels` sejam usados. |
| `verify_integrity` | Verifica se há duplicatas no novo eixo concatenado e gera uma exceção se houver; por padrão (`False`), permite duplicatas. |
| `ignore_index` | Não preserva os índices ao longo do eixo de concatenação; em vez disso, cria um novo índice com `range(total_length)`. |

### 2.5. Combinando Dados com Sobreposição

Existe outra situação de combinação de dados que não pode ser expressa como uma operação de mesclagem ou concatenação. Você pode ter dois conjuntos de dados com índices que se sobrepõem total ou parcialmente. Como exemplo motivador, considere a função `where` do NumPy, que executa o equivalente orientado a arrays de uma expressão `if-else`:

In [42]:
a = pd.Series([np.nan, 2.5, 0.0, 3.5, 4.5, np.nan],
              index = ['f', 'e', 'd', 'c', 'b', 'a'])

a

f    NaN
e    2.5
d    0.0
c    3.5
b    4.5
a    NaN
dtype: float64

In [43]:
b = pd.Series([0., np.nan, 2., np.nan, np.nan, 5.],
              index=['a', 'b', 'c', 'd', 'e', 'f'])

b

a    0.0
b    NaN
c    2.0
d    NaN
e    NaN
f    5.0
dtype: float64

In [44]:
np.where(pd.isna(a), b, a)

array([0. , 2.5, 0. , 3.5, 4.5, 5. ])

Aqui, sempre que os valores em `a` forem nulos, os valores de `b` serão selecionados; caso contrário, os valores não nulos de `a` serão selecionados. O uso de `numpy.where` não verifica se os rótulos dos índices estão alinhados ou não (e nem exige que os objetos tenham o mesmo comprimento); portanto, se você quiser alinhar os valores por índice, use o método `combine_first` da classe `Series`:

In [45]:
a.combine_first(b)

a    0.0
b    4.5
c    3.5
d    0.0
e    2.5
f    5.0
dtype: float64

Com DataFrames, a função `combine_first` faz a mesma coisa coluna por coluna, então você pode pensar nela como `"preenchendo"` os dados ausentes no objeto que a chama com os dados do objeto que você passa:

In [46]:
df1 = pd.DataFrame({'a': [1., np.nan, 5., np.nan],
                    'b': [np.nan, 2., np.nan, 6.],
                    'c': range(2, 18, 4)})

df1

,a,b,c
0,1.0,NaN,2
1,NaN,2.0,6
2,5.0,NaN,10
3,NaN,6.0,14


In [47]:
df2 = pd.DataFrame({'a': [5., 4., np.nan, 3., 7.],
                    'b': [np.nan, 3., 4., 6., 8.]})

df2

,a,b
0,5.0,NaN
1,4.0,3.0
2,NaN,4.0
3,3.0,6.0
4,7.0,8.0


In [48]:
df1.combine_first(df2)

,a,b,c
0,1.0,NaN,2.0
1,4.0,2.0,6.0
2,5.0,4.0,10.0
3,3.0,6.0,14.0
4,7.0,8.0,NaN


O resultado da função `combine_first` com objetos `DataFrame` será a união de todos os nomes das colunas.

## 3. Remodelagem e Rotação

Existem diversas operações básicas para reorganizar dados tabulares. Essas operações são chamadas de operações de remodelagem (*reshape*) ou rotação (*pivot*).

### 3.1. Remodelagem com Indexação Hierárquica

A indexação hierárquica fornece uma maneira consistente de reorganizar dados em um DataFrame.

Existem duas ações principais:

- `stack`: Esta ação `"rotaciona"` ou `"pivota"` os dados das colunas para as linhas.

- `unstack`: Esta ação `"pivota"` os dados das linhas para as colunas.

Ilustraremos essas operações por meio de uma série de exemplos. Considere um pequeno `DataFrame` com arrays de strings como índices de linha e coluna:

In [72]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(['Ohio', 'Colorado'], name='state'),
                    columns=pd.Index(['one', 'two', 'three'], name='number'))

data

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


Ao usar o método `stack` nesses dados, as colunas são transformadas em linhas, produzindo uma `Series`:

In [73]:
result = data.stack()

result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int64

A partir de uma `Series` hierarquicamente indexada, você pode reorganizar os dados de volta em um `DataFrame` com o método `unstack`:

In [74]:
result.unstack()

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


Por padrão, o nível mais interno é desempilhado (o mesmo acontece com a `stack`). Você pode desempilhar um nível diferente passando um número ou nome para ele:

In [75]:
result.unstack(level=0)

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


In [76]:
result.unstack(level='state')

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


O desempilhamento pode introduzir dados faltantes se nem todos os valores do nível forem encontrados em cada subgrupo:

In [77]:
s1 = pd.Series([0, 1, 2, 3], index=['a', 'b', 'c', 'd'], dtype='Int64')

s1

a    0
b    1
c    2
d    3
dtype: Int64

In [78]:
s2 = pd.Series([4, 5, 6], index=['c', 'd', 'e'], dtype='Int64')

s2

c    4
d    5
e    6
dtype: Int64

In [79]:
data2 = pd.concat([s1, s2], keys=['one', 'two'])

data2

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: Int64

O empilhamento filtra os dados ausentes por padrão, tornando a operação mais facilmente invertível:

In [80]:
data2.unstack()

,a,b,c,d,e
one,0,1,2,3,<NA>
two,<NA>,<NA>,4,5,6


In [81]:
data2.unstack().stack()

one  a       0
     b       1
     c       2
     d       3
     e    <NA>
two  a    <NA>
     b    <NA>
     c       4
     d       5
     e       6
dtype: Int64

Ao desempilhar elementos em um `DataFrame`, o nível desempilhado torna-se o nível mais baixo no resultado:

In [82]:
df = pd.DataFrame({'left': result, 'right': result + 5},
                  columns=pd.Index(['left', 'right'], name='side'))

df

side             left  right
state    number             
Ohio     one        0      5
         two        1      6
         three      2      7
Colorado one        3      8
         two        4      9
         three      5     10

In [83]:
df.unstack(level='state')

side   left          right         
state  Ohio Colorado  Ohio Colorado
number                             
one       0        3     5        8
two       1        4     6        9
three     2        5     7       10

Assim como em `unstack`, ao chamar `stack` podemos indicar o nome do eixo a ser empilhado:

In [84]:
df.unstack(level='state').stack(level='side', future_stack=True)

state         Ohio  Colorado
number side                 
one    left      0         3
       right     5         8
two    left      1         4
       right     6         9
three  left      2         5
       right     7        10

### 3.2. Pivotando do Formato "Longo" para o "Largo"

Uma forma comum de armazenar múltiplas séries temporais em bancos de dados e arquivos CSV é o que às vezes é chamado de formato longo (*long*) ou empilhado (*stacked*). Nesse formato, os valores individuais são representados por uma única linha em uma tabela, em vez de vários valores por linha.

Vamos carregar alguns dados de exemplo e realizar um pequeno processamento de séries temporais e outras operações de limpeza de dados:

In [88]:
data = pd.read_csv('./examples/macrodata.csv')
data = data.loc[:, ['year', 'quarter', 'realgdp', 'infl', 'unemp']]

data.head()

,year,quarter,realgdp,infl,unemp
0,1959,1,2710.349,0.00,5.8
1,1959,2,2778.801,2.34,5.1
2,1959,3,2775.488,2.74,5.3
3,1959,4,2785.204,0.27,5.6
4,1960,1,2847.699,2.31,5.2


Primeiro, utilizamos o `pandas.PeriodIndex` (que representa intervalos de tempo em vez de pontos no tempo), discutido com mais detalhes no `Módulo 7`, para combinar as colunas de ano e trimestre e definir o índice de forma a conter valores de data e hora no final de cada trimestre:

In [90]:
period_str = data['year'].astype(str) + 'Q' + data['quarter'].astype(str)
periods = pd.PeriodIndex(period_str, freq='Q', name='date')

periods

PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', name='date', length=203)

In [92]:
data.index = periods.to_timestamp('D')
data = data.drop(columns=['year', 'quarter'])

data.head()

,realgdp,infl,unemp
date,,,
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


Em seguida, selecionamos um subconjunto de colunas e atribuímos o nome `"item"` ao índice dessas colunas:

In [93]:
data = data.reindex(columns=['realgdp', 'infl', 'unemp'])
data.columns.name = 'item'

data.head()

item,realgdp,infl,unemp
date,,,
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


Por fim, remodelamos com `stack`, transformamos os novos níveis de índice em colunas com `reset_index` e, finalmente, damos à coluna que contém os valores dos dados o nome `"value"`:

In [94]:
long_data = (data.stack().reset_index().rename(columns={0: 'value'}))

long_data[:10]

,date,item,value
0,1959-01-01,realgdp,2710.349
1,1959-01-01,infl,0.000
2,1959-01-01,unemp,5.800
3,1959-04-01,realgdp,2778.801
4,1959-04-01,infl,2.340
5,1959-04-01,unemp,5.100
6,1959-07-01,realgdp,2775.488
7,1959-07-01,infl,2.740
8,1959-07-01,unemp,5.300
9,1959-10-01,realgdp,2785.204


Nesse formato longo para múltiplas séries temporais, cada linha da tabela representa uma única observação.

Os dados são frequentemente armazenados dessa forma em bancos de dados relacionais SQL, pois um esquema fixo (nomes de colunas e tipos de dados) permite que o número de valores distintos na coluna de itens mude à medida que novos dados são adicionados à tabela. No exemplo anterior, `date` e `item` geralmente seriam as chaves primárias (na terminologia de bancos de dados relacionais), oferecendo integridade relacional e facilitando as junções. Em alguns casos, os dados podem ser mais difíceis de trabalhar nesse formato; você pode preferir ter um `DataFrame` contendo uma coluna para cada valor distinto de `item` indexado por timestamps na coluna de `date`. O método `pivot` do `DataFrame` realiza exatamente essa transformação:

In [95]:
pivoted = long_data.pivot(index='date', columns='item', values='value')

pivoted.head()

item,infl,realgdp,unemp
date,,,
1959-01-01,0.00,2710.349,5.8
1959-04-01,2.34,2778.801,5.1
1959-07-01,2.74,2775.488,5.3
1959-10-01,0.27,2785.204,5.6
1960-01-01,2.31,2847.699,5.2


Os dois primeiros valores passados ​​são as colunas a serem usadas, respectivamente, como índice de linha e coluna, e, por fim, uma coluna de valores opcional para preencher o `DataFrame`. Suponha que você tenha duas colunas de valores que deseja remodelar simultaneamente:

In [96]:
long_data['value2'] = np.random.standard_normal( len(long_data) )

long_data[:10]

,date,item,value,value2
0,1959-01-01,realgdp,2710.349,-0.545102
1,1959-01-01,infl,0.000,-1.660459
2,1959-01-01,unemp,5.800,0.448698
3,1959-04-01,realgdp,2778.801,-1.318621
4,1959-04-01,infl,2.340,1.253587
5,1959-04-01,unemp,5.100,0.538068
6,1959-07-01,realgdp,2775.488,0.734723
7,1959-07-01,infl,2.740,-0.396877
8,1959-07-01,unemp,5.300,-0.737193
9,1959-10-01,realgdp,2785.204,-1.574505


Ao omitir o último argumento, você obtém um `DataFrame` com colunas hierárquicas:

In [97]:
pivoted = long_data.pivot(index='date', columns='item')

pivoted.head()

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8 -1.660459 -0.545102  0.448698
1959-04-01  2.34  2778.801   5.1  1.253587 -1.318621  0.538068
1959-07-01  2.74  2775.488   5.3 -0.396877  0.734723 -0.737193
1959-10-01  0.27  2785.204   5.6  1.290779 -1.574505 -0.497906
1960-01-01  2.31  2847.699   5.2  1.127308 -0.360231 -0.981506

In [98]:
pivoted['value'].head()

item,infl,realgdp,unemp
date,,,
1959-01-01,0.00,2710.349,5.8
1959-04-01,2.34,2778.801,5.1
1959-07-01,2.74,2775.488,5.3
1959-10-01,0.27,2785.204,5.6
1960-01-01,2.31,2847.699,5.2


Note que o comando `pivot` é equivalente a criar um índice hierárquico usando `set_index` seguido de uma chamada para `unstack`:

In [99]:
unstacked = long_data.set_index(['date', 'item']).unstack(level='item')

unstacked.head()

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8 -1.660459 -0.545102  0.448698
1959-04-01  2.34  2778.801   5.1  1.253587 -1.318621  0.538068
1959-07-01  2.74  2775.488   5.3 -0.396877  0.734723 -0.737193
1959-10-01  0.27  2785.204   5.6  1.290779 -1.574505 -0.497906
1960-01-01  2.31  2847.699   5.2  1.127308 -0.360231 -0.981506

### 3.3. Pivotando do Formato "Largo" para o "Longo"

Uma operação inversa à função `pivot` para DataFrames é o `pandas.melt`. Em vez de transformar uma coluna em várias em um novo `DataFrame`, ela mescla várias colunas em uma só, produzindo um `DataFrame` maior que o original. Vejamos um exemplo:

In [71]:
df = pd.DataFrame({"key": ["foo", "bar", "baz"],
                   "A": [1, 2, 3],
                   "B": [4, 5, 6],
                   "C": [7, 8, 9]})
df

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


A coluna `"key"` pode ser um indicador de grupo, e as outras colunas são valores de dados. Ao usar `pandas.melt`, devemos indicar quais colunas (se houver) são indicadores de grupo. Vamos usar `"key"` como o único indicador de grupo aqui:

In [72]:
melted = pd.melt(df, id_vars="key")
melted

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6
6,foo,C,7
7,bar,C,8
8,baz,C,9


Usando a função `pivot`, podemos retornar ao layout original:

In [73]:
reshaped = melted.pivot(index="key", 
                        columns="variable",
                        values="value")
reshaped

variable,A,B,C
key,,,
bar,2,5,8
baz,3,6,9
foo,1,4,7


Como o resultado da função `pivot` cria um índice a partir da coluna usada como rótulos de linha, podemos usar a função `reset_index` para mover os dados de volta para uma coluna:

In [74]:
reshaped.reset_index()

variable,key,A,B,C
0,bar,2,5,8
1,baz,3,6,9
2,foo,1,4,7


Você também pode especificar um subconjunto de colunas para usar como colunas de valor:

In [75]:
pd.melt(df, id_vars="key", value_vars=["A", "B"])

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6


A função `pandas.melt` também pode ser usado sem nenhum identificador de grupo:

In [76]:
pd.melt(df, value_vars=["A", "B", "C"])

,variable,value
0,A,1
1,A,2
2,A,3
3,B,4
4,B,5
5,B,6
6,C,7
7,C,8
8,C,9


In [77]:
pd.melt(df, value_vars=["key", "A", "B"])

,variable,value
0,key,foo
1,key,bar
2,key,baz
3,A,1
4,A,2
5,A,3
6,B,4
7,B,5
8,B,6


Agora que você já domina os conceitos básicos de Pandas para importação, limpeza e reorganização de dados, estamos prontos para avançar para a visualização de dados com matplotlib. Exploraremos outras áreas do Pandas mais adiante neste curso, quando abordarmos análises mais avançadas.

## 4. Exercícios

In [1]:
# Esta célula informa ao Jupyter para fornecer informações detalhadas de depuração
# quando ocorrer um erro de execução. Execute-o antes de trabalhar nos exercícios.

%xmode Verbose

Exception reporting mode: Verbose


### 4.1. Exercício

Crie uma `Series` com `MultiIndex`, faça seleções parciais (por nível externo e interno) e aplique `unstack`/`stack`.

Tarefas:
1. Selecione todas as entradas do rótulo externo `"b"`.
2. Selecione todas as entradas do nível interno `inner=2` (dica: `.loc[:, 2]`).
3. Aplique unstack (por padrão) e depois recupere a `Series` original com `stack`.
4. Faça `unstack(level="outer")` e descreva (em uma célula `markdown`) a mudança de eixos.

In [3]:
import numpy as np
import pandas as pd

# Criando uma Series com MultiIndex
rng = np.random.default_rng(42)
ex41 = pd.Series(rng.uniform(size=8),
                 index=pd.MultiIndex.from_arrays([["a","a","a","b","b","c","c","c"], 
                                                  [1,2,3,1,2,1,2,3]],
                                                 names=["outer","inner"]),
                 name="value")
print("Series Original:")
print(ex41)
print()
# Valores do rótulo externo b
print("Nível externo b:")
b = ex41["b"]
print(b)
print()
# Valores do rótulo interno 2
print("Seleção dos valores internos iguais a 2:")
i2 = ex41.loc[:, 2]
print(i2)
print()
# Transformando em DataFrame com Unstack e retornando ao modo original com Stack
unstacked = ex41.unstack()
print("Valores Unstacked:")
print(unstacked)
print()
stacked = unstacked.stack()
print("Valores que voltaram a serem uma Stack:")
print(stacked)
print()
# Unstack a partir do level 'outer', muda a disposição para coluna dos 'outer' e para linha dos 'inner'
print("Valores Unstacked a partir do level exterior")
outer_un = ex41.unstack(level="outer")
print(outer_un)

Series Original:
outer  inner
a      1        0.773956
       2        0.438878
       3        0.858598
b      1        0.697368
       2        0.094177
c      1        0.975622
       2        0.761140
       3        0.786064
Name: value, dtype: float64

Nível externo b:
inner
1    0.697368
2    0.094177
Name: value, dtype: float64

Seleção dos valores internos iguais a 2:
outer
a    0.438878
b    0.094177
c    0.761140
Name: value, dtype: float64

Valores Unstacked:
inner         1         2         3
outer                              
a      0.773956  0.438878  0.858598
b      0.697368  0.094177       NaN
c      0.975622  0.761140  0.786064

Valores que voltaram a serem uma Stack:
outer  inner
a      1        0.773956
       2        0.438878
       3        0.858598
b      1        0.697368
       2        0.094177
c      1        0.975622
       2        0.761140
       3        0.786064
dtype: float64

Valores Unstacked a partir do level exterior
outer         a         b    

### 4.2. Exercício

Troque a ordem dos níveis de um `MultiIndex` e ordene por nível específico, observando o impacto na exibição e na seleção.

Tarefas:
1. Use `.swaplevel("outer","inner")` e mostre o resultado.
2. Ordene por nível interno (`inner`) usando `.sort_index(level="inner")`.
3. Combine: troque os níveis e depois ordene pelo novo nível externo.

In [4]:
ex42 = ex41.copy()
display(ex42)

# 1. Swap de level interno e externo
swapped = ex42.swaplevel("outer", "inner")
display(swapped)
# 2. Ordenação pelo nível "inner"
ordered = swapped.sort_index(level="inner")
display(ordered)
# 3. Trocando novamente e ordenando pelo nível "outer"
normal = ordered.swaplevel("inner", "outer")
normal = normal.sort_index(level="outer")
display(normal)

outer  inner
a      1        0.773956
       2        0.438878
       3        0.858598
b      1        0.697368
       2        0.094177
c      1        0.975622
       2        0.761140
       3        0.786064
Name: value, dtype: float64

inner  outer
1      a        0.773956
2      a        0.438878
3      a        0.858598
1      b        0.697368
2      b        0.094177
1      c        0.975622
2      c        0.761140
3      c        0.786064
Name: value, dtype: float64

inner  outer
1      a        0.773956
       b        0.697368
       c        0.975622
2      a        0.438878
       b        0.094177
       c        0.761140
3      a        0.858598
       c        0.786064
Name: value, dtype: float64

outer  inner
a      1        0.773956
       2        0.438878
       3        0.858598
b      1        0.697368
       2        0.094177
c      1        0.975622
       2        0.761140
       3        0.786064
Name: value, dtype: float64

### 4.3. Exercício

Agregue por níveis de um `MultiIndex` usando `groupby(level=...)` para linhas e colunas.

Tarefas:
1. Agregue a soma por nível `'k'` (linhas).
2. Agregue a média por nível `'axis'` nas colunas (dica: use `.T.groupby(...).mean(...).T`).
3. Explique (em `markdown`) a diferença entre agrupar por `'k'` vs por `'axis'`.

In [24]:
ex43 = pd.DataFrame(rng.normal(size=(6,4)),
                    index=pd.MultiIndex.from_product([["A","B"], [1,2,3]], names=["grp","k"]),
                    columns=pd.MultiIndex.from_product([["X","Y"], ["u","v"]], names=["axis","comp"]))
display(ex43)

k = ex43.groupby("k").sum()
display(k)
col = ex43.T.groupby(level="axis").mean().T
display(col)

axis          X                   Y          
comp          u         v         u         v
grp k                                        
A   1 -0.933618 -0.205438 -0.950022 -0.339033
    2  0.840308 -1.727320  0.434424  0.237736
    3 -0.594150 -1.446058  0.072130 -0.529493
B   1  0.232676  0.021852  1.601779 -0.239356
    2 -1.023497  0.179276  0.219997  1.359188
    3  0.835111  0.356871  1.463303 -1.188763

axis         X                   Y          
comp         u         v         u         v
k                                           
1    -0.700941 -0.183585  0.651757 -0.578389
2    -0.183189 -1.548045  0.654420  1.596923
3     0.240961 -1.089187  1.535432 -1.718256

axis          X         Y
grp k                    
A   1 -0.569528 -0.644528
    2 -0.443506  0.336080
    3 -1.020104 -0.228682
B   1  0.127264  0.681212
    2 -0.422111  0.789592
    3  0.595991  0.137270

### 4.4. Exercício

Converta colunas em índices (incluindo `MultiIndex`) e depois traga de volta com `reset_index`.

Tarefas:
1. Faça `set_index(["b","c"])` e mantenha as colunas originais (`drop=False`).
2. Faça `reset_index` na saída anterior.
3. Explique (em `markdown`) quando é útil manter `drop=False`.

In [30]:
ex44 = pd.DataFrame({"a": range(6),
                     "b": list("aaabbb"),
                     "c": [1,1,2,2,3,3]})
display(ex44)
n44 = ex44.set_index(["b", "c"], drop=False)
display(n44)
reset = ex44.set_index(["b", "c"]).reset_index()
display(reset)

,a,b,c
0,0,a,1
1,1,a,1
2,2,a,2
3,3,b,2
4,4,b,3
5,5,b,3


a  b  c
b c         
a 1  0  a  1
  1  1  a  1
  2  2  a  2
b 2  3  b  2
  3  4  b  3
  3  5  b  3

,b,c,a
0,a,1,0
1,a,1,1
2,a,2,2
3,b,2,3
4,b,3,4
5,b,3,5


### 4.5. Exercício

Execute junções em colunas iguais e compare os resultados de `how`.

Tarefas:
1. Faça `merge` `inner` `on="key"` e conte as linhas resultantes por chave.
2. Repita com `how="left"` e `how="outer"` e compare onde aparecem `NaN`s.
3. Em `markdown`: explique por que muda a contagem de linhas para a chave `"b"`.

In [35]:
left = pd.DataFrame({"key": list("bbacaab"), "l": range(7)})
right = pd.DataFrame({"key": list("abd"), "r": [10,20,30]})
display(left, right)

merged = pd.merge(left, right, on="key")
display(merged)
print(f"Valores de chave: {merged["key"].value_counts()}")
left_merge = pd.merge(left, right, on="key", how="left")
display(left_merge)
print(f"Valores de chave em Left Join: {left_merge["key"].value_counts()}")
outer_merge = pd.merge(left, right, on="key", how="outer")
display(outer_merge)
print(f"Valores de chave em Outer Join: {outer_merge["key"].value_counts()}")

,key,l
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


,key,r
0,a,10
1,b,20
2,d,30


,key,l,r
0,b,0,20
1,b,1,20
2,a,2,10
3,a,4,10
4,a,5,10
5,b,6,20


Valores de chave: key
b    3
a    3
Name: count, dtype: int64


,key,l,r
0,b,0,20.0
1,b,1,20.0
2,a,2,10.0
3,c,3,NaN
4,a,4,10.0
5,a,5,10.0
6,b,6,20.0


Valores de chave em Left Join: key
b    3
a    3
c    1
Name: count, dtype: int64


,key,l,r
0,a,2.0,10.0
1,a,4.0,10.0
2,a,5.0,10.0
3,b,0.0,20.0
4,b,1.0,20.0
5,b,6.0,20.0
6,c,3.0,NaN
7,d,NaN,30.0


Valores de chave em Outer Join: key
a    3
b    3
c    1
d    1
Name: count, dtype: int64


### 4.6. Exercício

Realize `merge` com nomes de chaves diferentes; trate colisões de nomes e valide cardinalidade.

Tarefas:
1. Faça `merge` `l` (`left_on="lk"`) com `r` (`right_on="rk"`); resolva conflito de coluna `"v"` com `suffixes=("_L","_R")`.
2. Use `indicator=True` e conte quantas linhas são `left_only`, `right_only` e `both`.
3. Teste `validate="m:1"` e `"m:m"` e observe o comportamento/erros.

In [40]:
l = pd.DataFrame({"lk": list("bbaccb"), "v": range(6)})
r = pd.DataFrame({"rk": list("abda"), "v": [100,200,300,400]})
display(l, r)

merge = pd.merge(l, r, left_on="lk", right_on="rk", suffixes=("_L", "_R"))
display(merge)
merge_indicator = merge = pd.merge(l, r, left_on="lk", right_on="rk", suffixes=("_L", "_R"), indicator=True)
display(merge_indicator)
print(f"Contagem de valores a partir da contagem: {merge_indicator['_merge'].value_counts()}") # Como é Inner Join, todos são both

,lk,v
0,b,0
1,b,1
2,a,2
3,c,3
4,c,4
5,b,5


,rk,v
0,a,100
1,b,200
2,d,300
3,a,400


,lk,v_L,rk,v_R
0,b,0,b,200
1,b,1,b,200
2,a,2,a,100
3,a,2,a,400
4,b,5,b,200


,lk,v_L,rk,v_R,_merge
0,b,0,b,200,both
1,b,1,b,200,both
2,a,2,a,100,both
3,a,2,a,400,both
4,b,5,b,200,both


Contagem de valores a partir da contagem: _merge
both          5
left_only     0
right_only    0
Name: count, dtype: int64


### 4.7. Exercício

Faça junções usando índices (simples e `MultiIndex`) e compare `merge(..., left_index/right_index)` com `DataFrame.join`.

Tarefas:
1. Junte `li` e `ri` por índice (`outer`) com `merge` e com `join`; compare.
2. Faça merge `lcols` (`left_on=["g","k"]`) com `rmi` (`right_index=True`); repita com `how="outer"`.
3. Em `markdown`: quando preferir `.join` vs `pd.merge`?

In [6]:
# Índice simples
li = pd.DataFrame({"v": [1,2,3]}, index=list("abc"))
ri = pd.DataFrame({"w": [10,20]}, index=list("ab"))
display(li, ri)

# MultiIndex no lado direito
mi = pd.MultiIndex.from_arrays([list("AABBB"), [1,1,2,2,3]], names=["g","k"])
rmi = pd.DataFrame({"z": [5,6,7,8,9]}, index=mi)
lcols = pd.DataFrame({"g": list("AABAB"), "k": [1,2,1,3,2], "t": range(5)})
#display(lcols, rmi)

merge = pd.merge(li, ri, left_index=True, right_index=True, how="outer")
display(merge)
join = li.join(ri, how="outer")
display(join)

,v
a,1
b,2
c,3


,w
a,10
b,20


,v,w
a,1,10.0
b,2,20.0
c,3,NaN


,v,w
a,1,10.0
b,2,20.0
c,3,NaN


### 4.8. Exercício

Empilhe objetos com `pd.concat` variando `axis`, `keys`, `join` e `ignore_index`.

Tarefas:
1. Aplique `concat` em `[s1,s2,s3]` ao longo de linhas; repita com `axis="columns"` e `join="inner"`.
2. Aplique `concat` em `[df1, df2]` `axis="columns"` usando `keys=["L","R"]` e `names=["blk","col"]`.
3. Aplique `concat` em `[df1, df2]` ao longo de linhas com `ignore_index=True`.

In [43]:
s1 = pd.Series([0,1], index=["a","b"])
s2 = pd.Series([2,3,4], index=["c","d","e"])
s3 = pd.Series([5,6], index=["e","f"])  # note overlap em "e"
display(s1, s2, s3)

# 1 Concat ao longo das linhas
c = pd.concat([s1, s2, s3])
display(c)
# 1 Concat no eixo coluna
c_col = pd.concat([s1, s2, s3], axis='columns')
display(c_col)
# 1 Concat com join = 'inner'
c_inner = pd.concat([s1, s2, s3], axis='columns', join='inner')
display(c_inner)

df1 = pd.DataFrame(np.arange(6).reshape(3,2), index=list("abc"), columns=["x","y"])
df2 = pd.DataFrame(np.arange(4).reshape(2,2)+10, index=list("ac"), columns=["y","z"])
display(df1, df2)

# 2 Concat no eixo das colunas com índice hierárquico
dfc = pd.concat([df1, df2], axis='columns', keys=['L', 'R'], names=['blk', 'col'])
display(dfc)

# 3 Concat ao longo das linhas com ignore_index
dfcr = pd.concat([df1, df2], ignore_index=True)
display(dfcr)

a    0
b    1
dtype: int64

c    2
d    3
e    4
dtype: int64

e    5
f    6
dtype: int64

a    0
b    1
c    2
d    3
e    4
e    5
f    6
dtype: int64

,0,1,2
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,5.0
f,NaN,NaN,6.0


,0,1,2


,x,y
a,0,1
b,2,3
c,4,5


,y,z
a,10,11
c,12,13


blk  L        R      
col  x  y     y     z
a    0  1  10.0  11.0
b    2  3   NaN   NaN
c    4  5  12.0  13.0

,x,y,z
0,0.0,1,NaN
1,2.0,3,NaN
2,4.0,5,NaN
3,NaN,10,11.0
4,NaN,12,13.0


### 4.9. Exercício

Preencha valores ausentes de um objeto com os de outro alinhando por índice/colunas.

Tarefas:
1. Use `a.combine_first(b)` e explique o alinhamento por índice.
2. Use `d1.combine_first(d2)` e descreva como colunas/índices são unidos.
3. Refaça o passo `2` limitando colunas comuns previamente (ex.: `d1[["A","B"]]` `...`).

In [46]:
a = pd.Series([np.nan, 2.5, 0.0, np.nan], index=list("abcd"))
b = pd.Series([0.0, np.nan, 2.0, 5.0], index=list("acde"))
display(a, b)

df1 = pd.DataFrame({"A":[1.0, np.nan, 5.0, np.nan], "B":[np.nan, 2.0, np.nan, 6.0]}, index=list("abcd"))
df2 = pd.DataFrame({"A":[5.0, 4.0, np.nan], "B":[np.nan, 3.0, 4.0]}, index=list("bcd"))
display(df1, df2)

# 1 Combine de Series por índice
comb = a.combine_first(b)
display(comb)

# 2 Combine com DataFrame
df_comb = df1.combine_first(df2)
display(df_comb)

a    NaN
b    2.5
c    0.0
d    NaN
dtype: float64

a    0.0
c    NaN
d    2.0
e    5.0
dtype: float64

,A,B
a,1.0,NaN
b,NaN,2.0
c,5.0,NaN
d,NaN,6.0


,A,B
b,5.0,NaN
c,4.0,3.0
d,NaN,4.0


a    0.0
b    2.5
c    0.0
d    2.0
e    5.0
dtype: float64

,A,B
a,1.0,NaN
b,5.0,2.0
c,5.0,3.0
d,NaN,6.0


### 4.10. Exercício

Converta dados entre formatos `longo` e `largo`, incluindo múltiplas colunas de valores.

Tarefas:
1. Crie `long = wide.stack().reset_index().rename(columns={0:"value"})`.
2. Faça `pivot` de volta para o formato largo (`index="date"`, `columns="item"`, `values="value"`).
3. Adicione uma segunda coluna de valores (ex.: `"value2"` aleatória) ao `long` e faça `pivot` sem informar `values` (colunas hierárquicas).
4. Usando um `DataFrame` com colunas `"key"`,`"A"`,`"B"` e `"C"` use `pd.melt` para criar formato longo e depois retorne ao largo com `pivot` + `reset_index`.

In [91]:
# Dados largos -> longos
wide = pd.DataFrame({"date": pd.date_range("2020-01-01", periods=4, freq="QE"),
                     "realgdp": [1.0, 1.2, 1.1, 1.3],
                     "infl": [3.0, 2.8, 3.2, 3.1],
                     "unemp": [6.0, 5.8, 6.1, 6.0]}).set_index("date")
wide.columns.name = "item"
display(wide)

# 1 Display de dados em formato 'longo'
long = wide.stack().reset_index().rename(columns={0: 'value'})
display(long)

# 2 Retornar ao formato 'largo'
rewide = long.pivot(index="date", columns="item", values="value")
display(rewide)

# 3 Adicionar segunda coluna de valores e 'pivotar' sem values explícito
long["value2"] = np.random.standard_normal(len(long))
rewide = long.pivot(index="date", columns="item")
display(rewide)

# 4 Criar DataFrame com formato 'largo', 'melt' para o 'longo' e depois voltar ao 'largo' com 'pivot'
df = pd.DataFrame({"key": ["foo", "bar", "baz"],
                   "A": [1, 2, 3],
                   "B": [4, 5, 6],
                   "C": [7, 8, 9]})
display(df)
melted = pd.melt(df, id_vars="key")
display(melted)
rewide = melted.pivot(index="key", columns="variable", values="value").reset_index()
display(rewide)

item,realgdp,infl,unemp
date,,,
2020-03-31,1.0,3.0,6.0
2020-06-30,1.2,2.8,5.8
2020-09-30,1.1,3.2,6.1
2020-12-31,1.3,3.1,6.0


,date,item,value
0,2020-03-31,realgdp,1.0
1,2020-03-31,infl,3.0
2,2020-03-31,unemp,6.0
3,2020-06-30,realgdp,1.2
4,2020-06-30,infl,2.8
5,2020-06-30,unemp,5.8
6,2020-09-30,realgdp,1.1
7,2020-09-30,infl,3.2
8,2020-09-30,unemp,6.1
9,2020-12-31,realgdp,1.3


item,infl,realgdp,unemp
date,,,
2020-03-31,3.0,1.0,6.0
2020-06-30,2.8,1.2,5.8
2020-09-30,3.2,1.1,6.1
2020-12-31,3.1,1.3,6.0


value                  value2                    
item        infl realgdp unemp      infl   realgdp     unemp
date                                                        
2020-03-31   3.0     1.0   6.0  0.054355  0.399867  0.446918
2020-06-30   2.8     1.2   5.8  0.989159 -1.376962  1.067250
2020-09-30   3.2     1.1   6.1  2.257041 -0.600735 -0.058439
2020-12-31   3.1     1.3   6.0  0.615772 -1.418155  2.481357

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6
6,foo,C,7
7,bar,C,8
8,baz,C,9


variable,key,A,B,C
0,bar,2,5,8
1,baz,3,6,9
2,foo,1,4,7
